# Daily Challenge – Week 7, Day 1: Custom Attention Mechanism & SMS Spam Classification
Run in **Google Colab with a GPU** (Runtime → Change runtime type → GPU).

## Part 1: Setup & Data Loading

In [ ]:
!pip install --quiet datasets evaluate transformers[sentencepiece]

In [ ]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

# Load the UCI SMS Spam dataset from Hugging Face hub
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")

# Shuffle once so spam/ham are mixed in both splits
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Convert df to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split the data - 4,000 samples for training and 1,000 for validation
train_ds = hf_dataset.select(range(4000))
val_ds = hf_dataset.select(range(4000, 5000))

print(df["label"].value_counts())     # 0 = ham (normal), 1 = spam
df.head()

## Part 2: Tokenization Setup

In [ ]:
from transformers import GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default -- set it to eos
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

# Apply tokenization to both datasets
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

## Part 3: Pre-trained Model Setup
The classification head on top of GPT-2 starts with **random** weights, so we fine-tune it for one epoch on the training set before evaluating. Otherwise the "GPT-2" results would just be random guessing.

In [ ]:
import torch
from transformers import GPT2ForSequenceClassification, TrainingArguments, Trainer

model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2,
    pad_token_id=tokenizer.eos_token_id,
)

training_args = TrainingArguments(
    output_dir="gpt2-sms",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_tok.remove_columns(["sms"]).rename_column("label", "labels"))
trainer.train()

## Part 4: Custom Attention Implementation

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5          # 1 / sqrt(d)

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale   # (batch, seq, seq)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)        # each row sums to 1
        return torch.matmul(attn, value), attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)                          # (batch, seq, embed_dim)
        attn_output, _ = self.attn(embed, embed, embed)    # self-attention: Q = K = V
        pooled = attn_output.mean(dim=1)                   # average over the sequence
        return self.fc(pooled)

In [ ]:
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example["sms"],
        max_length=64,
        truncation=True,
        padding="max_length",
    )
    return {"input_ids": tokens, "label": example["label"]}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),
            'label': torch.tensor(item["label"], dtype=torch.long)
        }

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

In [ ]:
vocab_size = len(tokenizer)
embed_dim = 64
num_classes = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop (5 epochs instead of 1, so the small model has time to learn)
for epoch in range(5):
    attn_model.train()
    for batch in train_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = attn_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}: last batch loss {loss.item():.4f}")

print("Custom Attention model trained on SMS dataset. Sample batch loss:", loss.item())

## Part 5: Metrics & Evaluation

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall": recall.compute(predictions=preds, references=labels)["recall"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

In [ ]:
print("\n📊 Evaluating GPT-2 Model...")
gpt2_preds, gpt2_labels = [], []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    mask = torch.tensor(ex['attention_mask']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs, attention_mask=mask).logits
    gpt2_preds.append(torch.argmax(logits, dim=-1).cpu().item())
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    "accuracy": accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)["accuracy"],
    "precision": precision.compute(predictions=gpt2_preds, references=gpt2_labels)["precision"],
    "recall": recall.compute(predictions=gpt2_preds, references=gpt2_labels)["recall"],
    "f1": f1.compute(predictions=gpt2_preds, references=gpt2_labels)["f1"],
}
print("GPT-2 Metrics:", gpt2_metrics)

print("\n📊 Evaluating Custom Attention Model...")
attn_preds, attn_labels = [], []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
    attn_preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    "accuracy": accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],
    "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],
    "recall": recall.compute(predictions=attn_preds, references=attn_labels)["recall"],
    "f1": f1.compute(predictions=attn_preds, references=attn_labels)["f1"],
}
print("Attention Model Metrics:", attn_metrics)

pd.DataFrame({"GPT-2 (fine-tuned)": gpt2_metrics, "Custom attention": attn_metrics}).round(3)

## Part 6: Reflection Questions

**1. What are the roles of query, key, and value?**
Each token is turned into three vectors. The **query** represents *what this token is looking for* in the other tokens. The **keys** describe *what each token offers*, like labels used for matching: the dot product of a query with every key gives a relevance score. The **values** carry *the actual content* that gets passed on: the output for a token is a weighted average of all value vectors, weighted by those attention scores. In short: the query asks, the keys are matched, and the values are combined.

**2. Why do we use the scaling factor 1/√dₖ?**
A dot product sums dₖ products, so as the vector dimension grows, the scores get larger in size (their variance grows roughly with dₖ). Very large scores push the softmax into an extreme, nearly one-hot state, where one token gets ~100% of the attention and the gradients become tiny, so training slows or stalls. Dividing by √dₖ keeps the scores in a reasonable range, so the softmax stays smooth and gradients flow well.

**3. How does self-attention differ from RNNs?**
An RNN reads a sequence **one token at a time**, passing a hidden state along, so it can't be parallelised, and information from early tokens fades over long distances. Self-attention lets **every token look at every other token directly in one step**, so long-range dependencies are just one "hop" away, and all positions are computed **in parallel**, which makes training on GPUs much faster. The cost is that attention grows **quadratically** with sequence length (every pair of tokens), and it needs positional information added, since it has no built-in sense of order. Training is also more stable, without the vanishing gradients that affect long RNN chains.

**4. Performance analysis**
*(Compare the two columns in the metrics table above. Typical results are described here; check them against your run.)*

The **fine-tuned GPT-2** usually performs best, often around **98–99% accuracy with high precision and recall on spam**. It brings pre-trained knowledge of language from millions of documents, plus 12 Transformer layers with multi-head attention and positional information, so it understands phrases like "you've WON a free prize, call now" even with few examples. The **custom attention model** is tiny (one attention layer, no positional encoding, embeddings learned from scratch on 4,000 messages), but it still does surprisingly well on spam, often **mid-to-high 90s accuracy**, because spam has strong keyword signals ("free", "win", "txt", "£"). Its weaker point is usually **recall on spam**: with an imbalanced dataset (~13% spam), it tends to miss some spam messages.

**Trade-offs:** GPT-2 has ~124M parameters, needs a GPU, and is slower to train and run, but it's more accurate and robust. The custom model has a few million parameters (mostly embeddings), trains in seconds and is easy to inspect, but it understands much less context. **To improve the custom model:** add positional encodings, use multiple heads and stack 2–4 attention layers with feed-forward blocks and layer norm, ignore padding with an attention mask, use pre-trained word embeddings, train for more epochs, and handle the class imbalance with class weights.